# Sensitivity Analysis | Appendix

In [1]:
import pandas as pd

cases = pd.read_csv("sensitivity_margin_cases_both.csv").dropna(subset=["ref_margin"])
cases["group_median_margin"] = cases.groupby(["dataset", "model", "type"])["ref_margin"].transform("median")
cases["rel_margin"] = cases["ref_margin"] / cases["group_median_margin"]
cases["near_tie"] = cases["rel_margin"] < 0.5

In [2]:
MODEL_ORDER = ["cqd", "betae", "query2box", "gqe"]
MODEL_DISPLAY = {"cqd": "CQD", "betae": "BetaE", "query2box": "Query2Box", "gqe": "GQE"}

def diagnostics(sub):
    tied = sub["near_tie"]
    total_mass = sub["disagree_prob_k5"].sum()
    tie_mass = sub.loc[tied, "disagree_prob_k5"].sum()
    extreme = sub[sub["disagree_prob_k5"] > 0.5]
    rho = sub["rel_margin"].corr(sub["disagree_prob_k5"], method="spearman")
    return {
        "Near-tie pairs (%)": tied.mean() * 100,
        "Disagreement mass from near-ties (%)": tie_mass / total_mass * 100,
        "Mean disagreement, near-tie (%)": sub.loc[tied, "disagree_prob_k5"].mean() * 100,
        "Mean disagreement, clear-margin (%)": sub.loc[~tied, "disagree_prob_k5"].mean() * 100,
        "Extreme pairs, >50% disagreement (%)": len(extreme) / len(sub) * 100,
        "Spearman rho (margin vs. disagreement)": rho,
    }

# per (dataset, model) -- every column has exactly the same 220,000 query-answer
# pairs from Table~\ref{tab:query-stats} (verified below), since all four models
# are evaluated on the same set of pairs.
tables = {}
for ds in ["FB", "NELL"]:
    cols = {MODEL_DISPLAY[m]: diagnostics(cases[(cases["dataset"] == ds) & (cases["model"] == m)])
            for m in MODEL_ORDER}
    tables[ds] = pd.DataFrame(cols)
    print(f"--- {ds} ---")
    print(tables[ds].round(2))
    print()

n_pairs = cases[(cases["dataset"] == "FB") & (cases["model"] == "cqd")][["type", "query", "target"]].drop_duplicates().shape[0]
print(f"Sanity check: {n_pairs} query-answer pairs per (dataset, model) -- matches Table~\\ref{{tab:query-stats}}'s 220,000 total for FB15k-237+H.")

--- FB ---
                                          CQD  BetaE  Query2Box    GQE
Near-tie pairs (%)                      27.36  20.14      29.34  26.98
Disagreement mass from near-ties (%)    49.14  63.00      79.07  82.67
Mean disagreement, near-tie (%)         42.37  27.13      20.97  18.40
Mean disagreement, clear-margin (%)     16.51   4.02       2.30   1.43
Extreme pairs, >50% disagreement (%)     4.76   1.26       0.80   0.66
Spearman rho (margin vs. disagreement)  -0.85  -0.48      -0.61  -0.59



--- NELL ---
                                          CQD  BetaE  Query2Box    GQE
Near-tie pairs (%)                      28.59  30.24      36.99  37.85
Disagreement mass from near-ties (%)    54.32  65.50      68.28  70.66
Mean disagreement, near-tie (%)         38.36  23.22      22.49  21.49
Mean disagreement, clear-margin (%)     12.91   5.30       6.13   5.44
Extreme pairs, >50% disagreement (%)     3.00   1.64       1.99   1.70
Spearman rho (margin vs. disagreement)  -0.79  -0.48      -0.49  -0.52

Sanity check: 220000 query-answer pairs per (dataset, model) -- matches Table~\ref{tab:query-stats}'s 220,000 total for FB15k-237+H.


In [3]:
DS_DISPLAY = {"FB": "FB15k-237+H", "NELL": "NELL995+H"}

caption = (
    r"Concentration of $K{=}5$ disagreement in near-tie query-answer pairs (Section~\ref{sec:sensitivity}, "
    r"methodology in Appendix~\ref{sec:appendix-sensitivity-methodology}), per model, on FB15k-237+H and "
    r"NELL995+H "
    rf"({n_pairs:,} query-answer pairs per model per dataset, Table~\ref{{tab:query-stats}}). "
    r"\textit{Near-tie pairs}: share of pairs where the top two atoms' Shapley values are within "
    r"half of the typical margin for that model/query type/dataset. "
    r"\textit{Disagreement mass from near-ties}: share of all $K{=}5$ disagreement, summed across "
    r"every pair, contributed by near-tie pairs alone. "
    r"\textit{Mean disagreement, near-tie / clear-margin}: average $K{=}5$ disagreement rate "
    r"restricted to near-tie pairs, resp.\ all other (clear-margin) pairs. "
    r"\textit{Extreme pairs}: share of pairs that disagree with the $K{=}10$ reference more than "
    r"half the time. "
    r"\textit{Spearman rho}: correlation between a pair's margin and its disagreement rate "
    r"(more negative $=$ a smaller margin more reliably predicts disagreement)."
)

lines = [
    r"\begin{table}[t]",
    rf"\caption{{{caption}}}",
    r"\label{tab:sensitivity-diagnostics}",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\tabcolsep}{3pt}",
    r"\begin{tabular*}{\columnwidth}{@{}l@{\extracolsep{\fill}}rrrr@{}}",
    r"\toprule",
    r"\multirow{2}{*}{} & \multicolumn{4}{c}{\textbf{Model}} \\",
    r"\cmidrule(l){2-5}",
    r"& " + " & ".join(rf"\textbf{{{m}}}" for m in MODEL_DISPLAY.values()) + r" \\",
]
for ds in ["FB", "NELL"]:
    df = tables[ds]
    lines.append(r"\midrule")
    lines.append(rf"\multicolumn{{5}}{{c}}{{\textbf{{{DS_DISPLAY[ds]}}}}} \\")
    lines.append(r"\midrule")
    for name in df.index:
        vals = " & ".join(f"{df.loc[name, col]:.2f}" for col in df.columns)
        label = name.replace("%", "\\%")  # escape for LaTeX -- bare % starts a comment
        lines.append(f"{label} & {vals} \\\\")
lines += [r"\bottomrule", r"\end{tabular*}", r"\end{table}"]

tex = "\n".join(lines)
with open("table_sensitivity_diagnostics.tex", "w") as f:
    f.write(tex)
print(tex)

\begin{table}[t]
\caption{Concentration of $K{=}5$ disagreement in near-tie query-answer pairs (Section~\ref{sec:sensitivity}, methodology in Appendix~\ref{sec:appendix-sensitivity-methodology}), per model, on FB15k-237+H and NELL995+H (220,000 query-answer pairs per model per dataset, Table~\ref{tab:query-stats}). \textit{Near-tie pairs}: share of pairs where the top two atoms' Shapley values are within half of the typical margin for that model/query type/dataset. \textit{Disagreement mass from near-ties}: share of all $K{=}5$ disagreement, summed across every pair, contributed by near-tie pairs alone. \textit{Mean disagreement, near-tie / clear-margin}: average $K{=}5$ disagreement rate restricted to near-tie pairs, resp.\ all other (clear-margin) pairs. \textit{Extreme pairs}: share of pairs that disagree with the $K{=}10$ reference more than half the time. \textit{Spearman rho}: correlation between a pair's margin and its disagreement rate (more negative $=$ a smaller margin more r